# Processamento de textoPrepara **os três corpora de texto** do projeto, no mesmo lugar:| corpus | origem | papel ||---|---|---|| **reviews** | `reviews.csv` | a *entrega* — o que o hóspede relatou || **description + name** | `listings.csv` | a *promessa escrita* do anfitrião || **amenities** | `listings.csv` | a *promessa declarada*, estruturada e verificável |A separação importa: foto e descrição são duas promessas em modalidadesdiferentes, amenity é uma afirmação binária checável, e review é o contraponto.É isso que permite triangular em vez de só correlacionar duas pontas.**Saídas**- `data/processed/Airbnb/reviews_clean.csv`- `data/processed/Airbnb/listing_text.csv`- `data/processed/Airbnb/amenities.csv`

## 1. Setup

In [1]:
import ast
import re
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

RAW_REV = Path("../data/raw/Airbnb/reviews.csv")
RAW_LST = Path("../data/raw/Airbnb/listings.csv")
PROC    = Path("../data/processed/Airbnb")
PROC.mkdir(parents=True, exist_ok=True)

MIN_CHARS = 20     # ver seção 2.2
MIN_WORDS = 3
CHUNK     = 300_000

_BR   = re.compile(r"<br\s*/?>", re.I)
_WS   = re.compile(r"\s+")
_WORD = re.compile(r"\w+", re.UNICODE)
_AUTO = re.compile(r"automated posting|canceled this reservation|cancelled this reservation|"
                   r"cancelou esta reserva|anfitri.o cancelou|reservation was canceled", re.I)


def normalize(s: pd.Series) -> pd.Series:
    '''<br/> vira espaço, espaços colapsam, apara as pontas.

    Acentos e caixa PRESERVADOS: o tokenizador do modelo multilíngue foi
    treinado com eles, e remover piora o embedding.
    '''
    return (s.astype(str)
             .str.replace(_BR, " ", regex=True)
             .str.replace(_WS, " ", regex=True)
             .str.strip())


def n_words(s: pd.Series) -> pd.Series:
    return s.str.count(_WORD)

## 2. Reviews### 2.1 Diagnóstico

In [2]:
lens, words, n_total, n_null, n_auto = [], [], 0, 0, 0

for ch in pd.read_csv(RAW_REV, usecols=["listing_id", "comments"], chunksize=CHUNK, low_memory=False):
    n_total += len(ch)
    n_null  += int(ch.comments.isna().sum())
    txt = normalize(ch.comments.dropna())
    n_auto += int(txt.str.contains(_AUTO).sum())
    lens.append(txt.str.len().to_numpy()); words.append(n_words(txt).to_numpy())

L, W = np.concatenate(lens), np.concatenate(words)
print(f"linhas            : {n_total:,}")
print(f"comments nulos    : {n_null:,}")
print(f"msgs automáticas  : {n_auto:,}")
print("\npercentis de comprimento:",
      {f"p{p}": int(np.percentile(L, p)) for p in (10, 25, 50, 75, 90)})
print("\ncusto do corte (chars, com MIN_WORDS=3):")
for c in (0, 10, 20, 30, 50):
    k = (L >= c) & (W >= MIN_WORDS)
    print(f"   >= {c:3d}: {k.sum():>10,} ({100 * k.mean():.1f}%)")

linhas            : 1,577,515
comments nulos    : 113
msgs automáticas  : 16

percentis de comprimento: {'p10': 21, 'p25': 54, 'p50': 115, 'p75': 220, 'p90': 379}

custo do corte (chars, com MIN_WORDS=3):
   >=   0:  1,442,554 (91.5%)
   >=  10:  1,442,500 (91.4%)
   >=  20:  1,425,549 (90.4%)
   >=  30:  1,367,233 (86.7%)
   >=  50:  1,217,373 (77.2%)


### 2.2 Por que 20 caracteres e 3 palavras| faixa | exemplos | serve? ||---|---|---|| 1–10 | `"Recomendo"`, `"perfeito"`, `"ll"` | não — só polaridade || 10–20 | `"Muito bom!"`, `"Ótimo espaço."` | limítrofe || 20–30 | `"Espaço legal pertinho do masp"` | sim — nomeia aspecto |Abaixo de ~20 caracteres o review diz *que* o hóspede gostou, não *do quê*.O critério de palavras é o que manda: `MIN_WORDS=3` sozinho derruba ~8,5 %,e subir o corte de caracteres de 0 para 20 tira só ~1 % a mais.

In [3]:
stats = dict(lidos=0, nulos=0, auto=0, curtos=0, mantidos=0)
first = True

for ch in pd.read_csv(RAW_REV, chunksize=CHUNK, low_memory=False):
    stats["lidos"] += len(ch)
    n_antes = len(ch)
    ch = ch[ch.comments.notna()].copy()
    stats["nulos"] += n_antes - len(ch)

    ch["comments"] = normalize(ch.comments)
    is_auto = ch.comments.str.contains(_AUTO)
    stats["auto"] += int(is_auto.sum())
    ch = ch[~is_auto]

    keep = (ch.comments.str.len() >= MIN_CHARS) & (n_words(ch.comments) >= MIN_WORDS)
    stats["curtos"]   += int((~keep).sum())
    stats["mantidos"] += int(keep.sum())
    ch[keep].to_csv(PROC / "reviews_clean.csv", mode="w" if first else "a",
                    header=first, index=False)
    first = False

print({k: f"{v:,}" for k, v in stats.items()})
print(f"\nmantidos: {100 * stats['mantidos'] / stats['lidos']:.1f}%")

{'lidos': '1,577,515', 'nulos': '113', 'auto': '16', 'curtos': '151,853', 'mantidos': '1,425,533'}

mantidos: 90.4%


## 3. Textos do anúncio — a promessa escrita`description` (97,3 % preenchido, mediana 435 chars) é o par textual da foto de capa:as duas são marketing escolhido pelo anfitrião. `name` (100 %, mediana 43 chars) é aversão condensada — as poucas palavras que o host julga que mais vendem.> Checado e descartado neste export: `neighborhood_overview`, `license`,> `host_response_time/rate`, `host_acceptance_rate` e `instant_bookable` estão> **100 % vazios**. Não dá para usar, por mais atraentes que pareçam no schema.

In [4]:
lst = pd.read_csv(RAW_LST, low_memory=False,
                  usecols=["id", "name", "description", "amenities", "host_about",
                           "property_type", "room_type"])

for c in ["name", "description", "host_about"]:
    lst[c] = normalize(lst[c].fillna(""))
    lst.loc[lst[c].isin(["nan", ""]), c] = np.nan

txt = lst[["id", "name", "description", "host_about", "property_type", "room_type"]].copy()
txt["desc_chars"] = txt.description.str.len()
txt["desc_words"] = n_words(txt.description.fillna(""))
txt["has_desc"]   = txt.description.notna()

txt.to_csv(PROC / "listing_text.csv", index=False)

print(f"{len(txt):,} anúncios")
print(f"  com description : {100 * txt.has_desc.mean():.1f}%  (mediana {int(txt.desc_chars.median())} chars)")
print(f"  com host_about  : {100 * txt.host_about.notna().mean():.1f}%")
print("\n--- exemplo de description ---")
print(txt.description.dropna().iloc[0][:300])

42,354 anúncios
  com description : 97.3%  (mediana 418 chars)
  com host_about  : 54.2%

--- exemplo de description ---
Comfortable 2 bedrooms (1 master suite), 2 baths apt with terrace overlooking main swimming pool area. Only 1 block from Av. Paulista and 100m from Subway. Other amenities include: indoor/outdoor pools, saunas, gym, and much more. It's a paradise in the heart of São Paulo.


## 4. Amenities — a promessa declaradaMediana de **30 itens por anúncio**, 6.582 valores distintos no vocabulário.Dois tratamentos:1. **Matriz binária** das 40 mais frequentes — features diretas para qualquer modelo.2. **Lista verificável** — o subconjunto que dá para checar numa foto *e* num review.   É a âncora da triangulação: diferente de um score zero-shot, "declara piscina"   é um fato binário que não depende de prompt nenhum.

In [5]:
def parse(s):
    try:
        v = ast.literal_eval(s) if isinstance(s, str) and s.startswith("[") else []
        return [str(x).strip() for x in v]
    except Exception:
        return []

am = lst.amenities.apply(parse)
cnt = Counter(x for l in am for x in l)

print(f"vocabulário: {len(cnt):,} valores distintos")
print(f"itens por anúncio: mediana {int(am.apply(len).median())}, p90 {int(am.apply(len).quantile(.9))}")
print("\ntop 15:")
for k, v in cnt.most_common(15):
    print(f"  {100 * v / len(am):5.1f}%  {k[:46]}")

vocabulário: 6,577 valores distintos
itens por anúncio: mediana 30, p90 46

top 15:
   93.4%  Wifi
   91.2%  Kitchen
   72.7%  Hot water
   72.7%  TV
   70.3%  Microwave
   69.8%  Bed linens
   69.7%  Dishes and silverware
   67.7%  Hangers
   66.4%  Dedicated workspace
   66.1%  Cooking basics
   65.0%  Self check-in
   62.9%  Iron
   62.4%  Hair dryer
   57.7%  Elevator
   55.5%  Essentials


In [6]:
TOP_N = 40
top = [k for k, _ in cnt.most_common(TOP_N)]

def slug(s):
    s = re.sub(r"[^a-z0-9]+", "_", s.lower()).strip("_")
    return f"am_{s[:34]}"

A = pd.DataFrame({slug(k): am.apply(lambda l, k=k: int(k in l)) for k in top})
A.insert(0, "id", lst.id.to_numpy())
A["am_count"] = am.apply(len).to_numpy()

# --- verificáveis: casam por substring, para pegar variações do vocabulário ---
# (incluir, excluir) -- "Pool table" e bilhar, nao piscina, e aparece em 6,4%
# dos anuncios: sem o excludente ele contamina decl_pool inteiro.
VERIFIABLE = {
    "pool":      (["pool"], ["pool table"]),
    "gym":       (["gym", "exercise equipment"], []),
    "elevator":  (["elevator"], []),
    "balcony":   (["balcony", "patio"], []),
    "aircon":    (["air conditioning"], []),
    "workspace": (["dedicated workspace"], []),
    "parking":   (["free parking on premises", "paid parking"], []),
    "bbq":       (["bbq grill", "barbecue"], []),
    "cityview":  (["city skyline view", "city view"], ["pool view"]),
    "washer":    (["washer"], ["dishwasher"]),
}

def declares(items, inc, exc):
    for it in items:
        t = it.lower()
        if any(e in t for e in exc):
            continue
        if any(i in t for i in inc):
            return 1
    return 0

for k, (inc, exc) in VERIFIABLE.items():
    A[f"decl_{k}"] = am.apply(lambda l, i=inc, e=exc: declares(l, i, e)).to_numpy()

A.to_csv(PROC / "amenities.csv", index=False)
print(f"matriz: {A.shape}  -> {PROC / 'amenities.csv'}")
print("\n% que DECLARA cada amenity verificável:")
for k in VERIFIABLE:
    print(f"  {k:10s} {100 * A[f'decl_{k}'].mean():5.1f}%")

matriz: (42354, 52)  -> ../data/processed/Airbnb/amenities.csv

% que DECLARA cada amenity verificável:
  pool        46.6%
  gym         59.0%
  elevator    57.7%
  balcony     31.2%
  aircon      56.0%
  workspace   66.4%
  parking     49.7%
  bbq         14.5%
  cityview    15.9%
  washer      65.9%


## 5. Verificação

In [7]:
clean = pd.read_csv(PROC / "reviews_clean.csv", low_memory=False)
print("reviews_clean :", clean.shape, "|", f"{clean.listing_id.nunique():,} anúncios")
print("  min chars/palavras:", int(clean.comments.str.len().min()),
      "/", int(n_words(clean.comments).min()))
print("listing_text  :", txt.shape)
print("amenities     :", A.shape)

import glob, os
pilot = {int(os.path.basename(p)[:-4]) for p in glob.glob("../data/raw/Airbnb/photos/*.jpg")}
sub = clean[clean.listing_id.isin(pilot)]
print(f"\npiloto: {sub.listing_id.nunique()} anúncios, {len(sub):,} reviews")
print(f"  com description: {txt[txt.id.isin(pilot)].has_desc.sum()}")

reviews_clean : (1425533, 6) | 36,294 anúncios


  min chars/palavras: 20 / 3
listing_text  : (42354, 9)
amenities     : (42354, 52)

piloto: 294 anúncios, 31,754 reviews
  com description: 277
